#**Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistants:** Giuseppe Gallipoli (part 1) and Ali Yassine (part 2)

**Credits:** Moreno La Quatra

**Practice 1:** Text processing (part 1) and Topic modeling (part 2)

# **Part 2: Topic modeling**
---
Occurrence-based representations are high-dimensional, what is the dimension of the generated TF-IDF vector representation?
Topic modeling focuses on capturing latent topics in large document corpora.

The data collection used in this second part of the practice is provided [here](https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P1/CovidFake_filtered.csv) - [source: Zenodo](https://zenodo.org/record/4282522#.YVdCXcbOOpd)


# Exercise 7

Latent Semantic Indexing (LSI) models underlying concepts by using SVD (Singular Value Decomposition).

Use [gensim](https://radimrehurek.com/gensim/) library to:
1. Create a corpus composed of the headlines contained in the data collection.
2. Generate a [dictionary](https://radimrehurek.com/gensim/corpora/dictionary.html) to create a word -> id mapping (required by LSI module).
3. Using the dictionary, preprocess the corpus to obtain the representation required for LSI model training ([documentation here](https://radimrehurek.com/gensim/models/lsimodel.html)).
4. Inspect the top-5 topics generated by the LSI model for the analysed corpus.

In [ ]:
#!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P1/CovidFake_filtered.csv
!pip install gensim pandas

In [13]:
# your code here
import pandas as pd
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim.models import LsiModel

# load dataset
df = pd.read_csv("covidfake-filtered.csv")

# build corpus of headlines (as token lists)
texts = [simple_preprocess(str(s)) for s in df["headlines"].dropna().astype(str).tolist()]

# create gensim dictionary and bow corpus
dictionary = corpora.Dictionary(texts)
bow_corpus = [dictionary.doc2bow(text) for text in texts]

# print first document's bow representation
print(f"bow_corpus first document: {bow_corpus[:1]}")

# train LSI model (5 topics)
lsi = LsiModel(corpus=bow_corpus, id2word=dictionary, num_topics=5)

# print top-5 topics (showing top 10 words per topic)
topics = lsi.print_topics(num_topics=5, num_words=10)
for i, t in topics:
    print(f"Topic {i}: {t}")

bow_corpus first document: [[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 1), (19, 1), (20, 3), (21, 2), (22, 1), (23, 1), (24, 1), (25, 1)]]


/home/lookupmark/Documenti/github/polito/.venv/lib/python3.11/site-packages/gensim/models/lsimodel.py:963: DeprecationWarning: `scipy.sparse.sparsetools.csc_matvecs` is deprecated along with the `scipy.sparse.sparsetools` namespace. `scipy.sparse.sparsetools.csc_matvecs` will be removed in SciPy 1.14.0, and the `scipy.sparse.sparsetools` namespace will be removed in SciPy 2.0.0.
  sparsetools.csc_matvecs(


Topic 0: 0.711*"the" + 0.360*"of" + 0.305*"in" + 0.242*"to" + 0.174*"coronavirus" + 0.170*"and" + 0.147*"that" + 0.139*"covid" + 0.118*"is" + 0.089*"for"
Topic 1: 0.672*"in" + -0.588*"the" + 0.223*"to" + 0.201*"covid" + 0.133*"and" + 0.087*"video" + 0.084*"of" + 0.074*"has" + 0.066*"on" + 0.066*"people"
Topic 2: 0.668*"to" + -0.649*"of" + -0.141*"in" + 0.127*"is" + 0.120*"and" + 0.095*"coronavirus" + 0.077*"for" + 0.068*"be" + 0.052*"covid" + 0.051*"that"
Topic 3: 0.570*"of" + -0.502*"in" + 0.485*"to" + -0.237*"coronavirus" + -0.224*"the" + 0.206*"covid" + -0.072*"is" + -0.050*"was" + 0.046*"from" + -0.042*"new"
Topic 4: -0.481*"and" + -0.412*"covid" + -0.382*"that" + 0.375*"to" + -0.291*"is" + 0.250*"in" + 0.161*"coronavirus" + -0.156*"for" + 0.112*"of" + 0.095*"the"


# Exercise 8

The top-scored words contributing to each topic (if no stopword removal is applied) are english common words (e.g., *to, for, in, of, on*..). Moreover, missing punctuation removal could be critical for topic identification. Repeat the same procedure of Ex. 7 by adding preliminary preprocessing step to:
1. **remove stopwords**
2. **strip punctuation**
3. **lowercase all words**

In [14]:
# your code here
import re, string
import pandas as pd
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import LsiModel

# load dataset (re-read to make cell self-contained)
df = pd.read_csv("covidfake-filtered.csv")

def preprocess_text(text):
    txt = str(text).lower()                         # lowercase
    txt = re.sub(f'[{re.escape(string.punctuation)}]', ' ', txt)  # strip punctuation
    tokens = simple_preprocess(txt)                 # tokenize (already lowercased)
    tokens = [t for t in tokens if t not in STOPWORDS]  # remove stopwords
    return tokens

# build cleaned corpus of headlines
texts_clean = [preprocess_text(s) for s in df["headlines"].dropna().astype(str).tolist()]

# create gensim dictionary and bow corpus
dictionary = corpora.Dictionary(texts_clean)
bow_corpus = [dictionary.doc2bow(text) for text in texts_clean]

# inspect first cleaned document
print(f"cleaned tokens first doc: {texts_clean[:1]}")
print(f"bow_corpus first doc: {bow_corpus[:1]}")

# train LSI model (5 topics)
lsi = LsiModel(corpus=bow_corpus, id2word=dictionary, num_topics=5)

# print top-5 topics (top 10 words)
topics = lsi.print_topics(num_topics=5, num_words=10)
for i, t in topics:
    print(f"Topic {i}: {t}")

cleaned tokens first doc: [['post', 'claims', 'compulsory', 'vacination', 'violates', 'principles', 'bioethics', 'coronavirus', 'exist', 'pcr', 'test', 'returns', 'false', 'positives', 'influenza', 'vaccine', 'related', 'covid']]
bow_corpus first doc: [[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1)]]


/home/lookupmark/Documenti/github/polito/.venv/lib/python3.11/site-packages/gensim/models/lsimodel.py:963: DeprecationWarning: `scipy.sparse.sparsetools.csc_matvecs` is deprecated along with the `scipy.sparse.sparsetools` namespace. `scipy.sparse.sparsetools.csc_matvecs` will be removed in SciPy 1.14.0, and the `scipy.sparse.sparsetools` namespace will be removed in SciPy 2.0.0.
  sparsetools.csc_matvecs(


Topic 0: 0.734*"coronavirus" + 0.432*"covid" + 0.172*"video" + 0.141*"people" + 0.121*"shows" + 0.120*"facebook" + 0.108*"novel" + 0.107*"claim" + 0.097*"new" + 0.096*"shared"
Topic 1: 0.807*"covid" + -0.557*"coronavirus" + 0.055*"video" + -0.049*"novel" + 0.042*"shows" + -0.042*"new" + 0.037*"hospital" + 0.035*"claims" + 0.032*"lockdown" + -0.032*"china"
Topic 2: -0.364*"video" + -0.326*"facebook" + -0.296*"claim" + 0.284*"covid" + -0.283*"shows" + -0.277*"posts" + -0.266*"times" + 0.256*"coronavirus" + -0.249*"shared" + -0.195*"multiple"
Topic 3: -0.649*"video" + -0.298*"shows" + -0.250*"people" + 0.244*"facebook" + 0.240*"posts" + 0.219*"claim" + 0.206*"shared" + 0.179*"times" + 0.154*"multiple" + 0.142*"novel"
Topic 4: -0.888*"people" + 0.303*"video" + 0.129*"shows" + 0.091*"coronavirus" + 0.083*"covid" + -0.074*"government" + -0.074*"virus" + -0.066*"lockdown" + -0.064*"died" + 0.059*"patients"


# Exercise 9

Leveraging the same corpus used for LSI model generation, apply LDA modeling setting the number of topics to 5. Display the words most contributing to the those topics according to the LDA model.

In [15]:
# your code here
import re, string
import pandas as pd
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import LdaModel

# load dataset
df = pd.read_csv("covidfake-filtered.csv")

def preprocess_text(text):
    txt = str(text).lower()
    txt = re.sub(f'[{re.escape(string.punctuation)}]', ' ', txt)
    tokens = simple_preprocess(txt)
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

# build cleaned corpus (reuse same preprocessing as Ex.8)
texts_clean = [preprocess_text(s) for s in df["headlines"].dropna().astype(str).tolist()]

# create dictionary and bow corpus
dictionary = corpora.Dictionary(texts_clean)
# optional: filter extremes to remove very rare/very frequent tokens
dictionary.filter_extremes(no_below=5, no_above=0.5)
bow_corpus = [dictionary.doc2bow(text) for text in texts_clean]

# train LDA model with 5 topics
lda = LdaModel(corpus=bow_corpus, id2word=dictionary, num_topics=5, random_state=42, passes=10, alpha='auto', per_word_topics=False)

# print top words for each topic (top 10 words)
for idx, topic in lda.print_topics(num_topics=5, num_words=10):
    print(f"Topic {idx}: {topic}")

Topic 0: 0.025*"covid" + 0.024*"coronavirus" + 0.023*"government" + 0.018*"minister" + 0.016*"health" + 0.014*"gates" + 0.014*"india" + 0.014*"outbreak" + 0.013*"indian" + 0.012*"pandemic"
Topic 1: 0.035*"people" + 0.029*"coronavirus" + 0.024*"video" + 0.020*"covid" + 0.017*"shows" + 0.014*"lockdown" + 0.011*"photo" + 0.010*"italy" + 0.009*"quarantine" + 0.009*"india"
Topic 2: 0.105*"coronavirus" + 0.036*"covid" + 0.019*"virus" + 0.017*"new" + 0.016*"water" + 0.016*"cure" + 0.016*"novel" + 0.013*"prevent" + 0.011*"china" + 0.010*"vaccine"
Topic 3: 0.040*"coronavirus" + 0.039*"video" + 0.029*"shows" + 0.027*"facebook" + 0.025*"claim" + 0.023*"shared" + 0.018*"times" + 0.017*"posts" + 0.016*"covid" + 0.016*"people"
Topic 4: 0.060*"covid" + 0.057*"coronavirus" + 0.016*"new" + 0.016*"patients" + 0.015*"hospital" + 0.013*"china" + 0.012*"cases" + 0.010*"wuhan" + 0.010*"vaccine" + 0.009*"president"


# Exercise 10

Using [pyLDAvis]() library build an interactive visualization for the trained LDA model.

In [ ]:
!pip install pyLDAvis

In [20]:
# your code here
# gensim_models helper (preferred for recent pyLDAvis versions)
from pyLDAvis import gensim_models
import pyLDAvis
from IPython.display import display, HTML

# prepare visualization
pyLDAvis.enable_notebook()
vis = gensim_models.prepare(lda, bow_corpus, dictionary)

# display in notebook
display(vis)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4     -0.141002  0.018395       1        1  25.857720
1      0.151214 -0.084223       2        1  20.501427
2     -0.259123 -0.061475       3        1  20.132706
3      0.161271 -0.159678       4        1  18.604214
0      0.087641  0.286981       5        1  14.903932, topic_info=          Term        Freq       Total Category  logprob  loglift
22       video  995.000000  995.000000  Default  30.0000  30.0000
80       shows  717.000000  717.000000  Default  29.0000  29.0000
239   facebook  396.000000  396.000000  Default  28.0000  28.0000
235      claim  370.000000  370.000000  Default  27.0000  27.0000
249     shared  342.000000  342.000000  Default  26.0000  26.0000
..         ...         ...         ...      ...      ...      ...
61   president   92.370141  403.616191   Topic5  -4.8491   0.4289
91      public   54.391038   93.278313   Topic5  -5.3787   1.3642
198       said   77.316355  272.231205   Topic5  -5.0270   0.6448
283   american   48.111595   89.864060   Topic5  -5.5013   1.2788
83        says   49.799242  380.417575   Topic5  -5.4669  -0.1297

[314 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
1001      2  0.974968    airport
770       1  0.909893    alcohol
770       5  0.089708    alcohol
208       2  0.858934  allegedly
208       3  0.129428  allegedly
...     ...       ...        ...
755       5  0.063265      wuhan
289       1  0.911733      years
289       4  0.085810      years
2005      1  0.977547       york
1340      4  0.988963    youtube

[440 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[5, 2, 3, 4, 1])

# Exercise 11
**Credits:** Giuseppe Gallipoli

#### Introduction
[Large Language Models](https://en.wikipedia.org/wiki/Large_language_model) (LLMs) are a type of deep learning model capable of language generation. These models are built on deep learning architectures, primarily using neural networks, and are trained on massive amounts of text data. LLMs generally leverage the *Transformer* architecture (a groundbreaking deep architecture that you will study in more detail later in the course), which allows them to process language in context, capturing complex relationships between words and concepts.

Large Language Models have demonstrated excellent capabilities across a wide variety of tasks, making them versatile models which can be applied in diverse scenarios and use cases.

Given their relevance, although you have not yet covered this topic in the course, we will provide you, starting from this first laboratory practice, with practical applications showing how LLMs can be used to solve a diverse range of tasks.\
Don't worry about the theoretical or more technical aspects: they will be covered in more detail in due time.\
For now, the most important thing to know is that users interact with LLMs by means of a **prompt**, which is a piece of text containing the instruction or question the user wants to give or ask the model.

#### Topic modeling using Large Language Models

In this practice, we will use a Large Language Model to address a topic modeling-related task. Specifically, rather than modeling topic distributions as done with techniques like LSI or LDA, we will ask the LLM to extract the most relevant topic(s) from sentences (or from an entire corpus) according to different approaches.

For this task, we will use the [Phi](https://huggingface.co/microsoft/Phi-4-mini-instruct) 3.8B model, i.e., `microsoft/Phi-4-mini-instruct`.

\
<u>Suggestion</u>: To increase speed, switch to a GPU runtime. You can do this by clicking on Runtime → Change runtime type → Hardware accelerator → Select T4 GPU.\
If you encounter an `OutOfMemoryError`, try restarting the session by clicking on Runtime → Restart session.

In [ ]:
#!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P1/CovidFake_filtered.csv
!pip install transformers torch

  Using cached transformers-4.57.0-py3-none-any.whl.metadata (41 kB)
  Using cached torch-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2025.9.18-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.1.10-cp37-abi3-manylinu

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd, random

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# It may take some time to download the model
model_name = "microsoft/Phi-4-mini-instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# TODO: Load your dataset and extract a list of sentences to use as input.

sentences = pd.read_csv("covidfake-filtered.csv")["headlines"].dropna().astype(str).tolist()
print(f"Loaded {len(sentences)} sentences.")

This function `generate_topics` is designed to make it easy for you to explore
how a language model extracts topics from sentences.  

- You do **not** need to modify the function itself.
- In the upcoming steps, all you will do is **call this function** with a new
  prompt describing the topics you want the model to extract.
- The function will:
    1. Randomly pick a few sentences from the dataset.
    2. Send them to the model with your prompt.
    3. Print each sentence along with the predicted topics.
- Make sure your prompt clearly instructs the model on what format to return.

In [ ]:
def generate_topics(prompt, model, tokenizer, sentences, n_examples=5, device='cuda'):
    """
    Generate topics for random sentences using a language model.

    Args:
        prompt (str): Instruction for the model.
        model: The language model.
        tokenizer: The tokenizer for the model.
        sentences (list of str): Sentences to select from.
        n_examples (int): Number of sentences to process.
        device (str): Device to run the model on.

    Returns:
        list of (sentence, topics) tuples.
    """
    results = []
    for i, sentence in enumerate(random.choices(sentences, k=n_examples)):
        full_prompt = f"{prompt}{sentence}\nTopics: "
        inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

        output_ids = model.generate(**inputs, max_new_tokens=32)
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]

        if "Topics:" in decoded:
            topics = decoded.split("Topics:")[-1].strip()
        else:
            topics = decoded.strip()

        results.append((sentence, topics))
        print(f"Sentence {i+1}: {sentence}\nTopics: {topics}\n{'-'*50}")
    return results


**1<sup>st</sup> approach**: Ask the model to identify the topic(s) contained in a given sentence <u>without providing</u> a predefined list of topics to choose from.\
*Example of prompt*:\
Which are the most relevant topics of the following sentence?

In [ ]:
PROMPT1 = "Write your prompt here..."
generate_topics(PROMPT1, model, tokenizer, sentences, n_examples=10, device=device)

**2<sup>nd</sup> approach**: Ask the model to identify the topic(s) contained in a given sentence <u>providing</u> a predefined list of topics to choose from.

*Example of prompt*:\
Which are the most relevant topics of the following sentence?\
Choose among: medicine, COVID, Artificial Intelligence, treatment, English literature, vaccine, gardening

In [ ]:
PROMPT2 = """Write your prompt here..."""
generate_topics(PROMPT2, model, tokenizer, sentences, n_examples=10, device=device)

**3<sup>rd</sup> approach**: Ask the model to identify the topic(s) contained in a given sentence <u>providing</u> a predefined list of topics to choose from along with the corresponding definitions.

*Example of prompt*:\
Which are the most relevant topics of the following sentence?\
Choose among:
- medicine: treatment for illness or injury, or the study of this
- COVID: an infectious disease caused by a coronavirus
- Artificial Intelligence: computer systems that have some of the qualities that the human brain has, such as learn from data
- treatment: the use of drugs to cure a person of an illness or injury
- English literature: artistic works written in the English language, especially those with a high and lasting artistic value
- vaccine: a substance that is put into the body of a person or animal to protect them from a disease
- gardening: the job or activity of working in a garden

Definitions are taken from the [Cambridge Dictionary](https://dictionary.cambridge.org/) and have been slightly adapted.

In [ ]:
PROMPT3 = """Write your prompt here..."""
generate_topics(PROMPT3, model, tokenizer, sentences, n_examples=10, device=device)

After manually inspecting some of the outputs for each approach, here you can find some questions to reason about the results:
- Did you find an approach which worked best overall?
- Did you encounter any cases where the LLM failed?
- What happens if all the topics provided are irrelevant to the sentence?
- Does the presence of definitions improve the model's performance?
- What challenges or limitations did you observe?